# Day 7: Model Improvement

## Project: Retail Demand Forecasting & Inventory Optimization

### Objective
Improve the baseline LightGBM forecasting model and evaluate whether the proposed changes improve forecast accuracy.

### Baseline Performance
- MAE: 0.8838
- WAPE: 76.13%
- Test observations: 383,000
- Zero-sales observations: 231,349

### Day 7 Plan
1. Load the prepared test and training data.
2. Re-establish baseline model performance.
3. Investigate overprediction on zero-sales days.
4. Test targeted model improvements.
5. Compare results against the baseline.
6. Document findings and select the best-performing approach.

### Evaluation Metrics
- Mean Absolute Error (MAE)
- Weighted Absolute Percentage Error (WAPE)

**Important:** All model comparisons must use the same test period and evaluation observations.

### Load our prepared data

In [5]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path(r"C:\Users\lenovo\Retail-demand-forecasting")
DATA_PATH = PROJECT_DIR / "Data" / "processed" / "price_model_df.parquet"

price_model_df = pd.read_parquet(DATA_PATH)

print("Dataset loaded successfully!")
print("Shape:", price_model_df.shape)
print("\nColumns:")
print(price_model_df.columns.tolist())
print("\nFirst 5 rows:")
display(price_model_df.head())

Dataset loaded successfully!
Shape: (1913000, 34)

Columns:
['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd', 'sales', 'day_number', 'lag_1', 'lag_7', 'lag_28', 'rolling_mean_7', 'rolling_mean_28', 'date', 'weekday', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'snap_CA', 'snap_TX', 'snap_WI', 'day_of_week', 'month', 'is_event', 'naive_prediction', 'wm_yr_wk', 'sell_price', 'previous_price', 'price_change_pct', 'price_changed', 'log_price_change', 'price_missing']

First 5 rows:


,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,day_number,lag_1,...,month,is_event,naive_prediction,wm_yr_wk,sell_price,previous_price,price_change_pct,price_changed,log_price_change,price_missing
0,FOODS_1_003_TX_3_validation,FOODS_1_003,FOODS_1,FOODS,TX_3,TX,d_1,1,1,NaN,...,1,0,NaN,11101,2.88,NaN,NaN,0,NaN,0
1,FOODS_1_005_CA_1_validation,FOODS_1_005,FOODS_1,FOODS,CA_1,CA,d_1,3,1,NaN,...,1,0,NaN,11101,2.94,NaN,NaN,0,NaN,0
2,FOODS_1_005_TX_1_validation,FOODS_1_005,FOODS_1,FOODS,TX_1,TX,d_1,2,1,NaN,...,1,0,NaN,11101,2.94,NaN,NaN,0,NaN,0
3,FOODS_1_005_TX_3_validation,FOODS_1_005,FOODS_1,FOODS,TX_3,TX,d_1,2,1,NaN,...,1,0,NaN,11101,2.94,NaN,NaN,0,NaN,0
4,FOODS_1_008_CA_2_validation,FOODS_1_008,FOODS_1,FOODS,CA_2,CA,d_1,0,1,NaN,...,1,0,NaN,11101,NaN,NaN,NaN,0,NaN,1


In [6]:
from pathlib import Path

# Project directory
PROJECT_DIR = Path(r"C:\Users\lenovo\Retail-demand-forecasting")

print("Project directory exists:", PROJECT_DIR.exists())
print("\nFiles available in the project:\n")

for path in sorted(PROJECT_DIR.rglob("*")):
    if path.is_file() and (
        path.suffix.lower() in [".parquet", ".csv", ".pkl", ".joblib", ".json"]
    ):
        print(path.relative_to(PROJECT_DIR))

Project directory exists: True

Files available in the project:

Data\processed\price_model_df.parquet
Data\raw\m5-forecasting-accuracy\calendar.csv
Data\raw\m5-forecasting-accuracy\sales_train_evaluation.csv
Data\raw\m5-forecasting-accuracy\sales_train_validation.csv
Data\raw\m5-forecasting-accuracy\sample_submission.csv
Data\raw\m5-forecasting-accuracy\sell_prices.csv


### Check the feature Available in our prepared data 

In [7]:
# Day 7: Inspect the prepared modeling dataset

print("Prepared dataset shape:", price_model_df.shape)

print("\nAvailable columns:")
print(price_model_df.columns.tolist())

print("\nDate range:")
print("Start:", price_model_df["date"].min())
print("End:", price_model_df["date"].max())

print("\nTarget distribution:")
print(price_model_df["sales"].describe())

print("\nNumber of unique item-store series:")
print(price_model_df["id"].nunique())

print("\nMissing values in key columns:")
key_columns = [
    "sales",
    "lag_1",
    "lag_7",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_28",
    "sell_price",
    "log_price_change"
]

print(
    price_model_df[
        [col for col in key_columns if col in price_model_df.columns]
    ].isna().sum()
)


Prepared dataset shape: (1913000, 34)

Available columns:
['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd', 'sales', 'day_number', 'lag_1', 'lag_7', 'lag_28', 'rolling_mean_7', 'rolling_mean_28', 'date', 'weekday', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'snap_CA', 'snap_TX', 'snap_WI', 'day_of_week', 'month', 'is_event', 'naive_prediction', 'wm_yr_wk', 'sell_price', 'previous_price', 'price_change_pct', 'price_changed', 'log_price_change', 'price_missing']

Date range:
Start: 2011-01-29 00:00:00
End: 2016-04-24 00:00:00

Target distribution:
count    1.913000e+06
mean     9.925050e-01
std      3.175110e+00
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      1.000000e+00
max      6.340000e+02
Name: sales, dtype: float64

Number of unique item-store series:
1000

Missing values in key columns:
sales                    0
lag_1                 1000
lag_7                 7000
lag_28               28000
rolling_mean_7        7

In [8]:
from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\lenovo\Retail-demand-forecasting")

print("Searching for prepared datasets and model files...\n")

extensions = {".parquet", ".csv", ".pkl", ".pickle", ".joblib", ".feather"}

for path in sorted(PROJECT_DIR.rglob("*")):
    if path.is_file() and path.suffix.lower() in extensions:
        print(f"{path.relative_to(PROJECT_DIR)}  |  {path.stat().st_size / (1024**2):.2f} MB")

Searching for prepared datasets and model files...

Data\processed\price_model_df.parquet  |  10.06 MB
Data\raw\m5-forecasting-accuracy\calendar.csv  |  0.10 MB
Data\raw\m5-forecasting-accuracy\sales_train_evaluation.csv  |  116.10 MB
Data\raw\m5-forecasting-accuracy\sales_train_validation.csv  |  114.45 MB
Data\raw\m5-forecasting-accuracy\sample_submission.csv  |  4.99 MB
Data\raw\m5-forecasting-accuracy\sell_prices.csv  |  193.97 MB


### Recreate the chronological train/test split

In [10]:
import pandas as pd
import numpy as np

# Ensure date is in datetime format
price_model_df["date"] = pd.to_datetime(price_model_df["date"])

# Sort chronologically
price_model_df = price_model_df.sort_values(
    ["date", "id"]
).copy()

# Split by unique dates
unique_dates = sorted(price_model_df["date"].unique())
split_index = int(len(unique_dates) * 0.8)
split_date = unique_dates[split_index]

train_df = price_model_df[
    price_model_df["date"] < split_date
].copy()

test_df = price_model_df[
    price_model_df["date"] >= split_date
].copy()

print("Split date:", split_date)
print("Training shape:", train_df.shape)
print("Testing shape:", test_df.shape)

print("\nTraining date range:")
print(train_df["date"].min(), "to", train_df["date"].max())

print("\nTesting date range:")
print(test_df["date"].min(), "to", test_df["date"].max())

print("\nTarget observations:")
print("Training:", len(train_df))
print("Testing:", len(test_df))

Split date: 2015-04-08 00:00:00
Training shape: (1530000, 34)
Testing shape: (383000, 34)

Training date range:
2011-01-29 00:00:00 to 2015-04-07 00:00:00

Testing date range:
2015-04-08 00:00:00 to 2016-04-24 00:00:00

Target observations:
Training: 1530000
Testing: 383000


### Prepare features and handle missing values

In [11]:
# Define baseline features
features = [
    "lag_1",
    "lag_7",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_28",
    "day_of_week",
    "month",
    "is_event",
    "snap_CA",
    "snap_TX",
    "snap_WI"
]

target = "sales"

# Remove rows with missing lag/rolling features
train_clean = train_df.dropna(
    subset=features + [target]
).copy()

test_clean = test_df.dropna(
    subset=features + [target]
).copy()

# Create X and y
X_train = train_clean[features]
y_train = train_clean[target]

X_test = test_clean[features]
y_test = test_clean[target]

print("Feature preparation complete!")
print("\nTraining features:", X_train.shape)
print("Training target:", y_train.shape)
print("Testing features:", X_test.shape)
print("Testing target:", y_test.shape)

print("\nMissing values in X_train:", X_train.isna().sum().sum())
print("Missing values in X_test:", X_test.isna().sum().sum())

print("\nTest target distribution:")
print(y_test.describe())

print("\nZero-sales observations:", (y_test == 0).sum())
print("Zero-sales percentage:", round((y_test == 0).mean() * 100, 2), "%")

Feature preparation complete!

Training features: (1502000, 11)
Training target: (1502000,)
Testing features: (383000, 11)
Testing target: (383000,)

Missing values in X_train: 0
Missing values in X_test: 0

Test target distribution:
count    383000.000000
mean          1.160903
std           2.875120
min           0.000000
25%           0.000000
50%           0.000000
75%           1.000000
max          92.000000
Name: sales, dtype: float64

Zero-sales observations: 231349
Zero-sales percentage: 60.4 %


### Train the baseline LIGHTGBM model 

In [12]:
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error
import numpy as np

# Train baseline LightGBM model
lgb_model = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

print("Training LightGBM baseline...")

lgb_model.fit(X_train, y_train)

# Generate predictions
baseline_predictions = lgb_model.predict(X_test)

# Evaluate
baseline_mae = mean_absolute_error(
    y_test,
    baseline_predictions
)

baseline_wape = (
    np.abs(y_test.to_numpy() - baseline_predictions).sum()
    / y_test.sum()
) * 100

print("\nBaseline evaluation")
print("-" * 35)
print(f"MAE:  {baseline_mae:.4f}")
print(f"WAPE: {baseline_wape:.2f}%")
print(f"Test observations: {len(y_test):,}")
print(f"Zero-sales observations: {(y_test == 0).sum():,}")

Training LightGBM baseline...

Baseline evaluation
-----------------------------------
MAE:  0.8831
WAPE: 76.07%
Test observations: 383,000
Zero-sales observations: 231,349


### Investigate Zero-sales overpridiction

In [13]:
# Analyze predictions for zero-sales observations
zero_mask = y_test.to_numpy() == 0

zero_actual = y_test.to_numpy()[zero_mask]
zero_predictions = baseline_predictions[zero_mask]

print("ZERO-SALES ERROR ANALYSIS")
print("=" * 40)

print(f"Zero-sales observations: {zero_mask.sum():,}")
print(f"Percentage of test data: {zero_mask.mean() * 100:.2f}%")

print(f"\nAverage predicted sales: {zero_predictions.mean():.4f}")
print(f"Median predicted sales:  {np.median(zero_predictions):.4f}")
print(f"Minimum prediction:      {zero_predictions.min():.4f}")
print(f"Maximum prediction:      {zero_predictions.max():.4f}")

print(
    f"\nTotal predicted units on zero-sales days: "
    f"{zero_predictions.sum():,.2f}"
)

print(
    f"MAE on zero-sales observations: "
    f"{mean_absolute_error(zero_actual, zero_predictions):.4f}"
)

print(
    f"Negative predictions: "
    f"{(zero_predictions < 0).sum():,}"
)

ZERO-SALES ERROR ANALYSIS
Zero-sales observations: 231,349
Percentage of test data: 60.40%

Average predicted sales: 0.5037
Median predicted sales:  0.3157
Minimum prediction:      -2.3598
Maximum prediction:      36.1080

Total predicted units on zero-sales days: 116,534.81
MAE on zero-sales observations: 0.5041
Negative predictions: 8,104


#### Zero-sales overprediction: The model predicts an average of 0.5037 units when actual sales are zero, adding approximately 116,535 predicted units across zero-sales observations.
#### Negative predictions: 8,104 predictions are below zero, which is not meaningful for sales forecasting.

### Experiment-1 : Clip negative prediction

In [14]:
from sklearn.metrics import mean_absolute_error
import numpy as np

# Apply non-negative constraint to baseline predictions
clipped_predictions = np.maximum(baseline_predictions, 0)

# Evaluate original and clipped predictions
original_mae = mean_absolute_error(y_test, baseline_predictions)
clipped_mae = mean_absolute_error(y_test, clipped_predictions)

actual_sales = y_test.to_numpy()
original_wape = (
    np.abs(actual_sales - baseline_predictions).sum()
    / actual_sales.sum()
) * 100

clipped_wape = (
    np.abs(actual_sales - clipped_predictions).sum()
    / actual_sales.sum()
) * 100

print("EXPERIMENT 1: NON-NEGATIVE PREDICTIONS")
print("=" * 45)

print(f"Original MAE: {original_mae:.4f}")
print(f"Clipped MAE:  {clipped_mae:.4f}")
print(f"MAE change:   {clipped_mae - original_mae:+.4f}")

print(f"\nOriginal WAPE: {original_wape:.2f}%")
print(f"Clipped WAPE:  {clipped_wape:.2f}%")
print(f"WAPE change:   {clipped_wape - original_wape:+.2f} percentage points")

print(f"\nNegative predictions before: {(baseline_predictions < 0).sum():,}")
print(f"Negative predictions after:  {(clipped_predictions < 0).sum():,}")

EXPERIMENT 1: NON-NEGATIVE PREDICTIONS
Original MAE: 0.8831
Clipped MAE:  0.8830
MAE change:   -0.0001

Original WAPE: 76.07%
Clipped WAPE:  76.06%
WAPE change:   -0.01 percentage points

Negative predictions before: 8,275
Negative predictions after:  0


### Experiment-2:- Reduce zero-sales overpridiction

In [15]:
# Experiment 2: Reduce predictions by 10%, with a zero floor
adjustment_factor = 0.90

adjusted_predictions = np.maximum(
    clipped_predictions * adjustment_factor,
    0
)

actual_sales = y_test.to_numpy()

def evaluate_predictions(actual, predicted):
    mae = mean_absolute_error(actual, predicted)
    wape = (
        np.abs(actual - predicted).sum() / actual.sum() * 100
        if actual.sum() > 0 else np.nan
    )
    return mae, wape

# Overall performance
baseline_mae, baseline_wape = evaluate_predictions(
    actual_sales, clipped_predictions
)

adjusted_mae, adjusted_wape = evaluate_predictions(
    actual_sales, adjusted_predictions
)

# Performance on zero-sales and positive-sales observations
zero_mask = actual_sales == 0
positive_mask = actual_sales > 0

zero_mae_before, _ = evaluate_predictions(
    actual_sales[zero_mask], clipped_predictions[zero_mask]
)
zero_mae_after, _ = evaluate_predictions(
    actual_sales[zero_mask], adjusted_predictions[zero_mask]
)

positive_mae_before, positive_wape_before = evaluate_predictions(
    actual_sales[positive_mask], clipped_predictions[positive_mask]
)
positive_mae_after, positive_wape_after = evaluate_predictions(
    actual_sales[positive_mask], adjusted_predictions[positive_mask]
)

print("EXPERIMENT 2: 10% PREDICTION REDUCTION")
print("=" * 48)

print("\nOVERALL")
print(f"Baseline  | MAE: {baseline_mae:.4f} | WAPE: {baseline_wape:.2f}%")
print(f"Adjusted  | MAE: {adjusted_mae:.4f} | WAPE: {adjusted_wape:.2f}%")

print("\nZERO-SALES OBSERVATIONS")
print(f"Baseline MAE: {zero_mae_before:.4f}")
print(f"Adjusted MAE: {zero_mae_after:.4f}")

print("\nPOSITIVE-SALES OBSERVATIONS")
print(f"Baseline MAE: {positive_mae_before:.4f} | WAPE: {positive_wape_before:.2f}%")
print(f"Adjusted MAE: {positive_mae_after:.4f} | WAPE: {positive_wape_after:.2f}%")

EXPERIMENT 2: 10% PREDICTION REDUCTION

OVERALL
Baseline  | MAE: 0.8830 | WAPE: 76.06%
Adjusted  | MAE: 0.8670 | WAPE: 74.68%

ZERO-SALES OBSERVATIONS
Baseline MAE: 0.5039
Adjusted MAE: 0.4535

POSITIVE-SALES OBSERVATIONS
Baseline MAE: 1.4614 | WAPE: 49.84%
Adjusted MAE: 1.4977 | WAPE: 51.08%


### Compare multiple adjustment factors

In [16]:
# Compare different prediction adjustment factors
actual_sales = y_test.to_numpy()

adjustment_factors = [1.00, 0.95, 0.90, 0.85, 0.80]

results = []

for factor in adjustment_factors:
    predictions = np.maximum(clipped_predictions * factor, 0)

    overall_mae = mean_absolute_error(actual_sales, predictions)

    overall_wape = (
        np.abs(actual_sales - predictions).sum()
        / actual_sales.sum()
    ) * 100

    zero_mae = mean_absolute_error(
        actual_sales[zero_mask],
        predictions[zero_mask]
    )

    positive_mae = mean_absolute_error(
        actual_sales[positive_mask],
        predictions[positive_mask]
    )

    positive_wape = (
        np.abs(
            actual_sales[positive_mask]
            - predictions[positive_mask]
        ).sum()
        / actual_sales[positive_mask].sum()
    ) * 100

    results.append({
        "adjustment_factor": factor,
        "overall_MAE": overall_mae,
        "overall_WAPE_pct": overall_wape,
        "zero_sales_MAE": zero_mae,
        "positive_sales_MAE": positive_mae,
        "positive_sales_WAPE_pct": positive_wape
    })

adjustment_results = pd.DataFrame(results)

display(
    adjustment_results.round(4).sort_values("overall_MAE")
)

,adjustment_factor,overall_MAE,overall_WAPE_pct,zero_sales_MAE,positive_sales_MAE,positive_sales_WAPE_pct
4,0.80,0.8625,74.2977,0.4031,1.5634,53.3230
3,0.85,0.8633,74.3635,0.4283,1.5269,52.0778
2,0.90,0.8670,74.6794,0.4535,1.4977,51.0828
1,0.95,0.8735,75.2450,0.4787,1.4758,50.3375
0,1.00,0.8830,76.0629,0.5039,1.4614,49.8444


### Test a finer range

In [17]:
# Fine-grained adjustment-factor comparison
actual_sales = y_test.to_numpy()

adjustment_factors = [
    0.75, 0.775, 0.80, 0.825, 0.85,
    0.875, 0.90
]

fine_results = []

for factor in adjustment_factors:
    predictions = np.maximum(clipped_predictions * factor, 0)

    fine_results.append({
        "factor": factor,
        "overall_MAE": mean_absolute_error(
            actual_sales, predictions
        ),
        "overall_WAPE_pct": (
            np.abs(actual_sales - predictions).sum()
            / actual_sales.sum()
        ) * 100,
        "zero_sales_MAE": mean_absolute_error(
            actual_sales[zero_mask],
            predictions[zero_mask]
        ),
        "positive_sales_MAE": mean_absolute_error(
            actual_sales[positive_mask],
            predictions[positive_mask]
        )
    })

fine_results_df = pd.DataFrame(fine_results)

display(
    fine_results_df.round(4).sort_values("overall_MAE")
)

,factor,overall_MAE,overall_WAPE_pct,zero_sales_MAE,positive_sales_MAE
3,0.825,0.8625,74.2982,0.4157,1.5442
2,0.800,0.8625,74.2977,0.4031,1.5634
1,0.775,0.8633,74.3604,0.3905,1.5844
4,0.850,0.8633,74.3635,0.4283,1.5269
0,0.750,0.8647,74.4872,0.3779,1.6074
5,0.875,0.8648,74.4903,0.4409,1.5114
6,0.900,0.8670,74.6794,0.4535,1.4977


### Verify the best adjustment factor

In [18]:
# Show the three best factors by overall MAE
top_three = (
    fine_results_df
    .sort_values("overall_MAE")
    .head(3)
    .copy()
)

print("TOP 3 ADJUSTMENT FACTORS")
print("=" * 75)

display(top_three)

print("\nBest factor by MAE:", 
      fine_results_df.loc[fine_results_df["overall_MAE"].idxmin(), "factor"])

print("Best factor by WAPE:", 
      fine_results_df.loc[fine_results_df["overall_WAPE_pct"].idxmin(), "factor"])

TOP 3 ADJUSTMENT FACTORS


,factor,overall_MAE,overall_WAPE_pct,zero_sales_MAE,positive_sales_MAE
2,0.800,0.862525,74.297708,0.403110,1.563378
3,0.825,0.862530,74.298153,0.415707,1.544173
1,0.775,0.863252,74.360403,0.390513,1.584433



Best factor by MAE: 0.8
Best factor by WAPE: 0.8


### Save the final experiment

In [19]:
# Build a final experiment comparison table
final_comparison = pd.DataFrame([
    {
        "model": "LightGBM baseline (non-negative)",
        "adjustment_factor": 1.00,
        "overall_MAE": baseline_mae,
        "overall_WAPE_pct": baseline_wape,
        "zero_sales_MAE": zero_mae_before,
        "positive_sales_MAE": positive_mae_before
    },
    {
        "model": "LightGBM + 10% reduction",
        "adjustment_factor": 0.90,
        "overall_MAE": adjusted_mae,
        "overall_WAPE_pct": adjusted_wape,
        "zero_sales_MAE": zero_mae_after,
        "positive_sales_MAE": positive_mae_after
    },
    {
        "model": "LightGBM + best tested adjustment",
        "adjustment_factor": 0.80,
        "overall_MAE": fine_results_df.loc[
            fine_results_df["factor"] == 0.80, "overall_MAE"
        ].iloc[0],
        "overall_WAPE_pct": fine_results_df.loc[
            fine_results_df["factor"] == 0.80, "overall_WAPE_pct"
        ].iloc[0],
        "zero_sales_MAE": fine_results_df.loc[
            fine_results_df["factor"] == 0.80, "zero_sales_MAE"
        ].iloc[0],
        "positive_sales_MAE": fine_results_df.loc[
            fine_results_df["factor"] == 0.80, "positive_sales_MAE"
        ].iloc[0]
    }
])

display(final_comparison.round(4))

,model,adjustment_factor,overall_MAE,overall_WAPE_pct,zero_sales_MAE,positive_sales_MAE
0,LightGBM baseline (non-negative),1.0,0.8830,76.0629,0.5039,1.4614
1,LightGBM + 10% reduction,0.9,0.8670,74.6794,0.4535,1.4977
2,LightGBM + best tested adjustment,0.8,0.8625,74.2977,0.4031,1.5634


### Save your day 7 results

In [20]:
from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\lenovo\Retail-demand-forecasting")
REPORTS_DIR = PROJECT_DIR / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

output_file = REPORTS_DIR / "day7_model_improvement_results.csv"

final_comparison.to_csv(output_file, index=False)

print("Day 7 results saved successfully!")
print("File:", output_file)
print("\nSaved results:")
display(final_comparison.round(4))

Day 7 results saved successfully!
File: C:\Users\lenovo\Retail-demand-forecasting\reports\day7_model_improvement_results.csv

Saved results:


,model,adjustment_factor,overall_MAE,overall_WAPE_pct,zero_sales_MAE,positive_sales_MAE
0,LightGBM baseline (non-negative),1.0,0.8830,76.0629,0.5039,1.4614
1,LightGBM + 10% reduction,0.9,0.8670,74.6794,0.4535,1.4977
2,LightGBM + best tested adjustment,0.8,0.8625,74.2977,0.4031,1.5634


## Inventory Optimization

In [22]:
# Step 1: Calculate historical demand by item and store

inventory_df = price_model_df.copy()

# Aggregate historical sales for each item-store combination
demand_summary = (
    inventory_df
    .groupby(["id", "item_id", "store_id"], as_index=False)
    .agg(
        total_sales=("sales", "sum"),
        active_days=("sales", lambda x: (x > 0).sum()),
        observation_days=("sales", "count"),
        average_daily_demand=("sales", "mean"),
        maximum_daily_demand=("sales", "max")
    )
)

# Calculate the percentage of days with zero sales
demand_summary["zero_sales_pct"] = (
    1 - demand_summary["active_days"]
    / demand_summary["observation_days"]
) * 100

print("Inventory demand summary created!")
print("Item-store combinations:", len(demand_summary))

display(
    demand_summary.sort_values(
        "average_daily_demand",
        ascending=False
    ).head(10).round(2)
)

Inventory demand summary created!
Item-store combinations: 1000


,id,item_id,store_id,total_sales,active_days,observation_days,average_daily_demand,maximum_daily_demand,zero_sales_pct
304,FOODS_3_318_WI_3_validation,FOODS_3_318,WI_3,54770,1350,1913,28.63,367,29.43
444,FOODS_3_723_CA_3_validation,FOODS_3_723,CA_3,44780,1497,1913,23.41,122,21.75
283,FOODS_3_252_WI_3_validation,FOODS_3_252,WI_3,37221,1905,1913,19.46,98,0.42
823,HOUSEHOLD_1_521_CA_3_validation,HOUSEHOLD_1_521,CA_3,30266,1746,1913,15.82,72,8.73
761,HOUSEHOLD_1_277_CA_3_validation,HOUSEHOLD_1_277,CA_3,27759,1317,1913,14.51,107,31.16
56,FOODS_1_218_CA_3_validation,FOODS_1_218,CA_3,22230,1812,1913,11.62,92,5.28
295,FOODS_3_288_CA_2_validation,FOODS_3_288,CA_2,21951,963,1913,11.47,73,49.66
597,HOBBIES_1_371_CA_3_validation,HOBBIES_1_371,CA_3,19812,1507,1913,10.36,85,21.22
115,FOODS_2_181_TX_3_validation,FOODS_2_181,TX_3,18824,1850,1913,9.84,51,3.29
808,HOUSEHOLD_1_465_WI_2_validation,HOUSEHOLD_1_465,WI_2,17733,1287,1913,9.27,53,32.72


#### Calculate historical demand for each item-store combination. We'll use it as the foundation for inventory optimization.

In [24]:
# Summarize historical demand for inventory optimization

inventory_df = price_model_df.copy()

demand_summary = (
    inventory_df
    .groupby(["id", "item_id", "store_id"], as_index=False)
    .agg(
        total_sales=("sales", "sum"),
        active_days=("sales", lambda x: (x > 0).sum()),
        observation_days=("sales", "count"),
        average_daily_demand=("sales", "mean"),
        maximum_daily_demand=("sales", "max")
    )
)

demand_summary["zero_sales_pct"] = (
    (1 - demand_summary["active_days"] /
     demand_summary["observation_days"]) * 100
)

print("INVENTORY OPTIMIZATION")
print("=" * 40)
print("Item-store combinations:", len(demand_summary))

display(
    demand_summary
    .sort_values("average_daily_demand", ascending=False)
    .head(10)
    .round(2)
)

INVENTORY OPTIMIZATION
Item-store combinations: 1000


,id,item_id,store_id,total_sales,active_days,observation_days,average_daily_demand,maximum_daily_demand,zero_sales_pct
304,FOODS_3_318_WI_3_validation,FOODS_3_318,WI_3,54770,1350,1913,28.63,367,29.43
444,FOODS_3_723_CA_3_validation,FOODS_3_723,CA_3,44780,1497,1913,23.41,122,21.75
283,FOODS_3_252_WI_3_validation,FOODS_3_252,WI_3,37221,1905,1913,19.46,98,0.42
823,HOUSEHOLD_1_521_CA_3_validation,HOUSEHOLD_1_521,CA_3,30266,1746,1913,15.82,72,8.73
761,HOUSEHOLD_1_277_CA_3_validation,HOUSEHOLD_1_277,CA_3,27759,1317,1913,14.51,107,31.16
56,FOODS_1_218_CA_3_validation,FOODS_1_218,CA_3,22230,1812,1913,11.62,92,5.28
295,FOODS_3_288_CA_2_validation,FOODS_3_288,CA_2,21951,963,1913,11.47,73,49.66
597,HOBBIES_1_371_CA_3_validation,HOBBIES_1_371,CA_3,19812,1507,1913,10.36,85,21.22
115,FOODS_2_181_TX_3_validation,FOODS_2_181,TX_3,18824,1850,1913,9.84,51,3.29
808,HOUSEHOLD_1_465_WI_2_validation,HOUSEHOLD_1_465,WI_2,17733,1287,1913,9.27,53,32.72


### Calculate Demand Variablity

In [25]:
# Calculate demand variability for inventory planning

demand_variability = (
    inventory_df
    .groupby(["id", "item_id", "store_id"])["sales"]
    .agg(
        average_daily_demand="mean",
        daily_demand_std="std",
        median_daily_demand="median",
        maximum_daily_demand="max"
    )
    .reset_index()
)

# Replace undefined standard deviations with zero
demand_variability["daily_demand_std"] = (
    demand_variability["daily_demand_std"].fillna(0)
)

# Coefficient of variation: relative demand variability
demand_variability["demand_cv"] = np.where(
    demand_variability["average_daily_demand"] > 0,
    demand_variability["daily_demand_std"]
    / demand_variability["average_daily_demand"],
    np.nan
)

print("Demand variability calculated!")
print("Item-store combinations:", len(demand_variability))

display(
    demand_variability
    .sort_values("demand_cv", ascending=False)
    .head(10)
    .round(3)
)

Demand variability calculated!
Item-store combinations: 1000


,id,item_id,store_id,average_daily_demand,daily_demand_std,median_daily_demand,maximum_daily_demand,demand_cv
474,HOBBIES_1_006_CA_4_validation,HOBBIES_1_006,CA_4,0.013,0.138,0.0,3,10.598
14,FOODS_1_079_CA_2_validation,FOODS_1_079,CA_2,0.014,0.146,0.0,2,10.327
49,FOODS_1_186_CA_4_validation,FOODS_1_186,CA_4,0.016,0.132,0.0,2,8.444
660,HOBBIES_2_110_CA_4_validation,HOBBIES_2_110,CA_4,0.026,0.214,0.0,4,8.363
490,HOBBIES_1_056_CA_4_validation,HOBBIES_1_056,CA_4,0.052,0.436,0.0,8,8.333
653,HOBBIES_2_082_CA_4_validation,HOBBIES_2_082,CA_4,0.025,0.204,0.0,4,8.318
647,HOBBIES_2_070_CA_4_validation,HOBBIES_2_070,CA_4,0.021,0.164,0.0,2,7.823
679,HOUSEHOLD_1_020_WI_2_validation,HOUSEHOLD_1_020,WI_2,0.051,0.395,0.0,7,7.719
168,FOODS_2_341_CA_2_validation,FOODS_2_341,CA_2,0.037,0.280,0.0,5,7.640
664,HOBBIES_2_119_CA_4_validation,HOBBIES_2_119,CA_4,0.023,0.170,0.0,4,7.373


### Calculate safety stock and reorder points

In [26]:
# Calculate safety stock and reorder points

LEAD_TIME_DAYS = 7
SERVICE_LEVEL_Z = 1.645  # Approx. 95% one-sided service level

inventory_policy = demand_variability.copy()

inventory_policy["lead_time_demand"] = (
    inventory_policy["average_daily_demand"] * LEAD_TIME_DAYS
)

inventory_policy["safety_stock"] = (
    SERVICE_LEVEL_Z
    * inventory_policy["daily_demand_std"]
    * np.sqrt(LEAD_TIME_DAYS)
).apply(np.ceil)

inventory_policy["reorder_point"] = np.ceil(
    inventory_policy["lead_time_demand"]
    + inventory_policy["safety_stock"]
).astype(int)

print("Inventory policy calculated!")
print("Assumed lead time:", LEAD_TIME_DAYS, "days")
print("Target service level: 95%")
print("Products analyzed:", len(inventory_policy))

display(
    inventory_policy[
        [
            "id",
            "store_id",
            "average_daily_demand",
            "daily_demand_std",
            "lead_time_demand",
            "safety_stock",
            "reorder_point"
        ]
    ]
    .sort_values("reorder_point", ascending=False)
    .head(10)
    .round(2)
)

Inventory policy calculated!
Assumed lead time: 7 days
Target service level: 95%
Products analyzed: 1000


,id,store_id,average_daily_demand,daily_demand_std,lead_time_demand,safety_stock,reorder_point
304,FOODS_3_318_WI_3_validation,WI_3,28.63,35.76,200.41,156.0,357
444,FOODS_3_723_CA_3_validation,CA_3,23.41,20.13,163.86,88.0,252
283,FOODS_3_252_WI_3_validation,WI_3,19.46,11.33,136.20,50.0,187
761,HOUSEHOLD_1_277_CA_3_validation,CA_3,14.51,14.38,101.58,63.0,165
823,HOUSEHOLD_1_521_CA_3_validation,CA_3,15.82,11.27,110.75,50.0,161
295,FOODS_3_288_CA_2_validation,CA_2,11.47,15.12,80.32,66.0,147
56,FOODS_1_218_CA_3_validation,CA_3,11.62,11.11,81.34,49.0,131
597,HOBBIES_1_371_CA_3_validation,CA_3,10.36,10.73,72.50,47.0,120
808,HOUSEHOLD_1_465_WI_2_validation,WI_2,9.27,9.74,64.89,43.0,108
305,FOODS_3_319_TX_1_validation,TX_1,8.07,9.54,56.52,42.0,99


### Calculate recommended ordeer quantities

In [27]:
# Calculate illustrative order quantities

TARGET_COVERAGE_DAYS = 14

inventory_policy["target_stock"] = np.ceil(
    inventory_policy["average_daily_demand"]
    * (LEAD_TIME_DAYS + TARGET_COVERAGE_DAYS)
).astype(int)

inventory_policy["recommended_order_qty"] = (
    inventory_policy["target_stock"]
    - inventory_policy["reorder_point"]
).clip(lower=0)

print("Illustrative order quantities calculated!")
print("Target coverage:", TARGET_COVERAGE_DAYS, "days")
print("Products analyzed:", len(inventory_policy))

display(
    inventory_policy[
        [
            "id",
            "store_id",
            "average_daily_demand",
            "safety_stock",
            "reorder_point",
            "target_stock",
            "recommended_order_qty"
        ]
    ]
    .sort_values("recommended_order_qty", ascending=False)
    .head(10)
    .round(2)
)

Illustrative order quantities calculated!
Target coverage: 14 days
Products analyzed: 1000


,id,store_id,average_daily_demand,safety_stock,reorder_point,target_stock,recommended_order_qty
304,FOODS_3_318_WI_3_validation,WI_3,28.63,156.0,357,602,245
444,FOODS_3_723_CA_3_validation,CA_3,23.41,88.0,252,492,240
283,FOODS_3_252_WI_3_validation,WI_3,19.46,50.0,187,409,222
823,HOUSEHOLD_1_521_CA_3_validation,CA_3,15.82,50.0,161,333,172
761,HOUSEHOLD_1_277_CA_3_validation,CA_3,14.51,63.0,165,305,140
56,FOODS_1_218_CA_3_validation,CA_3,11.62,49.0,131,245,114
115,FOODS_2_181_TX_3_validation,TX_3,9.84,24.0,93,207,114
386,FOODS_3_570_CA_3_validation,CA_3,9.25,29.0,94,195,101
597,HOBBIES_1_371_CA_3_validation,CA_3,10.36,47.0,120,218,98
307,FOODS_3_327_CA_3_validation,CA_3,9.04,31.0,95,190,95


### Save the inventory policy

In [28]:
# Save inventory optimization outputs

from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\lenovo\Retail-demand-forecasting")
REPORTS_DIR = PROJECT_DIR / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

output_path = REPORTS_DIR / "day8_inventory_policy.csv"

inventory_policy.to_csv(output_path, index=False)

print("Inventory policy saved successfully!")
print("Products:", len(inventory_policy))
print("File:", output_path)

print("\nSummary statistics:")
display(
    inventory_policy[
        [
            "average_daily_demand",
            "safety_stock",
            "reorder_point",
            "target_stock",
            "recommended_order_qty"
        ]
    ].describe().round(2)
)

Inventory policy saved successfully!
Products: 1000
File: C:\Users\lenovo\Retail-demand-forecasting\reports\day8_inventory_policy.csv

Summary statistics:


,average_daily_demand,safety_stock,reorder_point,target_stock,recommended_order_qty
count,1000.00,1000.00,1000.00,1000.00,1000.00
mean,0.99,6.58,14.04,21.33,7.52
std,1.98,8.93,22.47,41.62,19.51
min,0.01,1.00,2.00,1.00,0.00
25%,0.18,3.00,5.00,4.00,0.00
50%,0.41,4.00,8.00,9.00,2.00
75%,0.99,7.00,15.00,21.00,7.00
max,28.63,156.00,357.00,602.00,245.00
